The notebook does the following :
* Trains and tracks experiments and model versions using MLflow. Trains three models with hyperparameter tuning-
    - Logistic Regression: Interpretable and a strong baseline for linear text classification.
    - Naive Bayes Classifier: A probabilistic model effective on high-dimensional sparse text data.
    - Linear SVM: Maximizes class separation and often achieves high performance in text classification tasks.
    - Feature extraction: Raw text is converted into numerical features using TF-IDF (Term Frequency–Inverse Document Frequency). The `TfidfVectorizer` is embedded within a scikit-learn `Pipeline`, ensuring that feature extraction is learned from the training data and consistently applied to validation and test sets during model training and evaluation.
      - TF-IDF (Term Frequency–Inverse Document Frequency) converts text into numerical features by weighting words based on their importance within a document and across the corpus, emphasizing discriminative terms while downweighting common ones.
      - Unigram TF-IDF offers a strong baseline with lower dimensionality and sparsity, enabling efficient training and better generalization for linear models compared to higher-order n-grams.
* Builds, tunes, and registers three benchmark text-classification models.
* Compares models using AUCPR and selects the best-performing benchmark.
    - `Area Under the Precision–Recall Curve (AUCPR)` measures how well a model identifies the positive class, especially when data is imbalanced.
* Enables versioned retrieval and evaluation of registered models across experiments.
* saves the best model


### Required Libraries

In [2]:
!pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.0/814.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 5.6 MB/s eta 0:00:00


In [3]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import joblib
import os

### Upload Data

We are procceeding with latest data split corresponsing to `seed = 123`.
Upload train.csv, test.cs and validation.csv.

In [4]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving validation.csv to validation.csv


### Helper Functions

In [5]:
# Load dataset splits
def load_split(file_path: str) -> pd.DataFrame:
    return pd.read_csv(file_path)

# Fit model to training data
def fit_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    return model

# AUCPR Metric
def compute_aucpr(y_true, y_scores):
    return average_precision_score(y_true, y_scores)

# Evaluate model with multiple metrics
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }
# Build model
def build_model(model_name, param):

    if model_name == "LogisticRegression":
        clf = LogisticRegression(C=param, max_iter=1000)

    elif model_name == "NaiveBayes":
        clf = MultinomialNB(alpha=param)

    elif model_name == "LinearSVM":
        clf = LinearSVC(C=param)

    else:
        raise ValueError("Unknown model")

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("clf", clf)
    ])

    return pipeline
# Hyperparameter tuning using GridSearchCV | F1 Score on validation set
def tune_and_evaluate_model(
    model_name,
    param_values,
    X_train, y_train,
    X_val, y_val
):
    records = []

    for param in param_values:
        if model_name == "LogisticRegression":
            model = Pipeline([
                ("tfidf", TfidfVectorizer()),
                ("clf", LogisticRegression(C=param, max_iter=1000))
            ])

        elif model_name == "NaiveBayes":
            model = Pipeline([
                ("tfidf", TfidfVectorizer()),
                ("clf", MultinomialNB(alpha=param))
            ])

        elif model_name == "LinearSVM":
            model = Pipeline([
                ("tfidf", TfidfVectorizer()),
                ("clf", LinearSVC(C=param))
            ])

        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)

        train_metrics = compute_metrics(y_train, train_pred)
        val_metrics = compute_metrics(y_val, val_pred)

        records.append({
            "param": param,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "model": model
        })

    df = pd.DataFrame(records)

    best_row = df.loc[df["val_f1"].idxmax()]
    best_model = best_row["model"]

    return df, best_model, best_row

### Training and Evaluation

Updated to save best model.

In [7]:
# -------- Load data --------
train_df = load_split("train.csv")
val_df   = load_split("validation.csv")
test_df  = load_split("test.csv")

X_train, y_train = train_df["text"], train_df["label"]
X_val,   y_val   = val_df["text"],   val_df["label"]
X_test,  y_test  = test_df["text"],  test_df["label"]

model_configs = {
    "LogisticRegression": [0.1, 0.5, 1, 5, 10],
    "NaiveBayes":         [0.1, 0.5, 1.0, 1.5, 2.0],
    "LinearSVM":          [0.1, 0.5, 1,   5,   10],
}

best_models = {}

# -------- Hyperparameter tuning --------
print("\n===== Hyperparameter Tuning =====")

for model_name, param_values in model_configs.items():

    best_f1    = -1
    best_model = None
    best_param = None

    print(f"\n{model_name}")

    for param in param_values:
        model = build_model(model_name, param)
        model.fit(X_train, y_train)

        val_pred    = model.predict(X_val)
        val_metrics = compute_metrics(y_val, val_pred)

        print(f"  Param={param:<4} | Val F1={val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_f1:
            best_f1    = val_metrics["f1"]
            best_model = model
            best_param = param

    best_models[model_name] = best_model
    print(f"  -> Selected Param={best_param} (F1={best_f1:.4f})")

# -------- MLflow tracking --------
mlflow.set_experiment("sms_spam_models")

test_metrics_summary = {}
os.makedirs("models", exist_ok=True)

print("\n===== Experiment Tracking & Model Versioning =====")

for model_name, model in best_models.items():

    with mlflow.start_run(run_name=model_name):

        test_pred = model.predict(X_test)
        test_metrics = compute_metrics(y_test, test_pred)

        if hasattr(model, "decision_function"):
            test_scores = model.decision_function(X_test)
        else:
            test_scores = model.predict_proba(X_test)[:, 1]

        aucpr = compute_aucpr(y_test, test_scores)

        mlflow.log_metrics(test_metrics)
        mlflow.log_metric("test_aucpr", aucpr)

        # Save each model to disk and log as artifact
        model_path = f"models/{model_name}.joblib"
        joblib.dump(model, model_path)
        mlflow.log_artifact(model_path)

        mlflow.sklearn.log_model(
            model,
            artifact_path=model_name,
            registered_model_name=model_name
        )

        test_metrics_summary[model_name] = {
            **test_metrics,
            "aucpr": aucpr
        }

        print(f"{model_name} registered | AUCPR={aucpr:.4f}")

# -------- Final comparison & save best model --------
print("\n===== Final Model Comparison =====")

best_model_name = max(
    test_metrics_summary,
    key=lambda m: test_metrics_summary[m]["aucpr"]
)

for model_name, metrics in test_metrics_summary.items():
    marker = "  <- BEST" if model_name == best_model_name else ""
    print(f"{model_name:<20} AUCPR={metrics['aucpr']:.4f}{marker}")

# Save best model as best_model.joblib (used by score.py and app.py)
joblib.dump(best_models[best_model_name], "best_model.joblib")
print(f"\nBest model ({best_model_name}) saved to 'best_model.joblib'")


===== Hyperparameter Tuning =====

LogisticRegression


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


  Param=0.1  | Val F1=0.0000
  Param=0.5  | Val F1=0.7182
  Param=1    | Val F1=0.8683
  Param=5    | Val F1=0.9358
  Param=10   | Val F1=0.9498
  -> Selected Param=10 (F1=0.9498)

NaiveBayes
  Param=0.1  | Val F1=0.9561
  Param=0.5  | Val F1=0.9550
  Param=1.0  | Val F1=0.9151
  Param=1.5  | Val F1=0.8557
  Param=2.0  | Val F1=0.7766
  -> Selected Param=0.1 (F1=0.9561)

LinearSVM
  Param=0.1  | Val F1=0.9005
  Param=0.5  | Val F1=0.9593
  Param=1    | Val F1=0.9640
  Param=5    | Val F1=0.9640
  Param=10   | Val F1=0.9686
  -> Selected Param=10 (F1=0.9686)

===== Experiment Tracking & Model Versioning =====


2026/03/08 13:03:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 13:03:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LogisticRegression'.
Created version '1' of model 'LogisticRegression'.
2026/03/08 13:03:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogisticRegression registered | AUCPR=0.9679


2026/03/08 13:03:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'NaiveBayes'.
Created version '1' of model 'NaiveBayes'.
2026/03/08 13:03:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


NaiveBayes registered | AUCPR=0.9708


2026/03/08 13:03:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearSVM registered | AUCPR=0.9729

===== Final Model Comparison =====
LogisticRegression   AUCPR=0.9679
NaiveBayes           AUCPR=0.9708
LinearSVM            AUCPR=0.9729  <- BEST

Best model (LinearSVM) saved to 'best_model.joblib'


Successfully registered model 'LinearSVM'.
Created version '1' of model 'LinearSVM'.


- Logistic Regression: At low values of the hyperparameter, the model underfits and misses spam (low recall); increasing it improves recall and F1 until performance stabilizes, with no strong signs of overfitting (plateau).
- Naive Bayes: Higher smoothing leads to underfitting, causing recall and F1 to drop because the model becomes too conservative in predicting spam.
- Linear SVM: Achieves high performance quickly and shows minimal train–validation gap, indicating good generalization.
- Precision is consistently high because the dataset is imbalanced (only ~13% spam), so models are cautious about predicting the minority class, leading to very few false positives but making recall the main challenge.